### Web Scraping Articles

In [1]:
import httpx
import pandas as pd
import numpy as np
import re
from selectolax.parser import HTMLParser
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch



def scrape_article(url):
    response = httpx.get(url)
    html = HTMLParser(response.text)
    
    title = html.css_first('title').text(strip=True) if html.css_first('title') else 'No title'
    paragraphs = [node.text(strip=True) for node in html.css('p') if node.text()]
    
    return {
        "title": title,
        "content": "\n\n".join(paragraphs)
    }



# Execute
url = "https://en.wikipedia.org/wiki/Ottoman_Empire" #https://www.ign.com/articles/donkey-kong-bananza-is-the-switch-2s-true-successor-to-super-mario-odyssey

article = scrape_article(url)
print("Title:", article["title"])
print("Content:", article["content"])

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Title: Ottoman Empire - Wikipedia
Content: 

TheOttoman Empire[l](/ˈɒtəmən/ⓘ), also called theTurkish Empire,[24][25]was animperial realm[m]that controlled much ofSoutheast Europe,West Asia, andNorth Africafrom the 14th to early 20th centuries; it also controlled parts of southeasternCentral Europe, between the early 16th and early 18th centuries.[26][27][28]

The empire emerged from abeylik, orprincipality, founded in northwesternAnatoliainc.1299by theTurkomantribal leaderOsman I. His successorsconqueredmuch of Anatolia and expanded into theBalkansby the mid-14th century, transforming theirpetty kingdominto a transcontinental empire. The Ottomans ended theByzantine Empirewith theconquest of Constantinoplein 1453 byMehmed II. With its capital atConstantinopleand control over a significant portion of theMediterranean Basin, the Ottoman Empire was at the centre of interactions between theMiddle Eastand Europe for six centuries. Ruling over so many peoples, the empire granted varying leve

In [2]:
type(article)

dict

### Sentiment Analysis

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device = 'mps')

In [ ]:
# chunking
def split_text(text, max_words=200): # set level of granularity (too low = not enough context, more rows in df)
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

chunks = split_text(article["content"]) # break up article text into chunks


# run classifier on each chunk and store labels and confidence
results = [] 
for i, chunk in enumerate(chunks):
    result = classifier(chunk, truncation=True)[0] # run BERT model on chunk (w/ truncation)
    
    results.append({
        "chunk_index": i,
        "text_snippet": chunk,
        "label": result['label'],
        "confidence": result['score']
    })


results = pd.DataFrame(results)
results

In [ ]:
confident_results = results[results['confidence'] >= 0.95]
confident_results['label'].value_counts()

### Text-Generation

In [ ]:
generator = pipeline('text-generation',
                     model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
                     device='mps')

In [ ]:
input_query = "Who is Laura Fryer?"

messages = [
    {"role": "system",
    "content": f"{article['content']}"},

    {"role": "user", "content": f"{input_query}"},
]

prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = generator(prompt, max_new_tokens=256, do_sample=True, temperature=0.3, top_k=50, top_p=0.95)

In [ ]:
response = outputs[0]['generated_text'].split('<|assistant|>')[-1].strip()

print(response)

In [ ]:
# Follow-up

input_query = "How qualified is this person to comment on this topic?"

messages = [
    {"role": "system",
    "content": f"{response}"},
    
    {"role": "user", "content": f"{input_query}"},
]

prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = generator(prompt, max_new_tokens=256, do_sample=True, temperature=0.1, top_k=50, top_p=0.95)
response = outputs[0]['generated_text'].split('<|assistant|>')[-1].strip()

print(response)

### RAG Chat-Loop

In [3]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from sklearn.preprocessing import normalize

# Setup
generator = pipeline('text-generation', model='TinyLlama/TinyLlama-1.1B-Chat-v1.0', device='mps')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Chunking
def split_text(text, max_words=200):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

# Build vector index once
chunks = split_text(article["content"])
doc_embeddings = normalize(embedding_model.encode(chunks))
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))



# Chat loop (dynamic follow-ups)
messages = []       # initialize message history

def chat(input_query):
    global messages

    query_embedding = normalize(embedding_model.encode([input_query]))
    D, I = index.search(np.array(query_embedding), k=3)
    retrieved_docs = [chunks[i] for i in I[0]]
    retrieved_context = "\n".join(retrieved_docs)

    if not messages:
        messages.append({
            "role": "system",
            "content": f"Use the following context to answer the user's question:\n{retrieved_context}"
        })

    messages.append({"role": "user", "content": input_query})

    prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = generator(prompt, max_new_tokens=256, do_sample=True, temperature=0.5, top_k=50, top_p=0.95)
    response = outputs[0]['generated_text'].split('<|assistant|>')[-1].strip()

    messages.append({"role": "assistant", "content": response})
    return response

In [4]:
# Example usage
chat("Who founded the empire?")

'The Ottoman Empire was founded by theTurkish tribes who migrated from Central Asia and the Middle East on a much larger scale. The first ruler of the Ottoman Empire was Osman I, who founded the empire in 1299 in northwesternAnatolia. The empire was initially ruled by theTurkishMillets, or petty kingdoms, who governed themselves under the rule of the sultan. The sultan was the head of state, but the empire was governed by a system of local governors calledtimarîs. The empire continued to maintain a flexible and strong economy, society, and military until the reigns of Selim I and Suleiman the Magnificent in the 16th century.'

In [5]:
chat("How long did he live for?")

'The Ottoman Empire was founded by Osman I in 1299 in northwesternAnatolia. Osman I ruled the empire until his death in 1322, after which his son, Osman II, became the ruler of the empire. Osman II ruled for 38 years until his death in 1354, at which point his son, Murad II, took over as the ruler of the empire. Murad II ruled for 21 years until his death in 1374, after which his son, Mehmed II, took over as the ruler of the empire. Mehmed II ruled for 37 years until his death in 1421, after which his son, Selim I, took over as the ruler of the empire. Selim I ruled for 12 years until his death in 1444, after which his son, Suleiman the Magnificent, took over as the ruler of the empire. Suleiman the Magnificent ruled for 42 years until his death in 1566, after which his son, Selim II, took over as'